# Exploratory Data Analysis - Amazon Reviews Dataset
## Mining Reviewer Rings: A Forensic Pipeline for Coordinated Fraud

This notebook performs exploratory data analysis on the Amazon Reviews 2023 Dataset (Appliances category) to understand the data structure, distributions, and patterns before implementing the fraud detection pipeline.

## 1. Setup and Imports

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
from pathlib import Path
from collections import Counter

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create output directory for figures
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

print("Libraries imported successfully!")
print(f"Polars version: {pl.__version__}")

## 2. Load Data

Loading the Amazon Reviews 2023 dataset (Appliances category):
- **Appliances.jsonl**: Reviews with text, ratings, timestamps, and verification status
- **meta_Appliances.jsonl**: Product metadata

In [ ]:
data_dir = Path('Appliances')

if not data_dir.exists():
    raise FileNotFoundError(f"Data directory not found: {data_dir}")

print("Available files:")
for file in data_dir.glob('*.jsonl'):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"  {file.name}: {size_mb:.2f} MB")

### 2.1 Load Review Data

In [ ]:
print("Loading Amazon review data...")
print("This may take a minute for 2M+ reviews...\n")

# Load all reviews from JSONL file
all_reviews = pl.read_ndjson(data_dir / 'Appliances.jsonl')

print(f"Total reviews loaded: {len(all_reviews):,}")
print(f"\nReview schema:")
print(all_reviews.schema)
print(f"\nSample reviews:")
all_reviews.head(3)

In [ ]:
# Standardize column names to match pipeline expectations
# Amazon: rating, text, asin, user_id, timestamp, verified_purchase
# Pipeline: stars, text, business_id, user_id, datetime

print("Standardizing column names...")

reviews_raw = all_reviews.select([
    pl.col('user_id'),
    pl.col('asin').alias('business_id'),  # product ID = business ID
    pl.col('rating').alias('stars'),
    pl.col('text'),
    pl.col('title').alias('review_title'),
    pl.col('timestamp'),
    pl.col('verified_purchase'),
    pl.col('helpful_vote'),
]).with_columns([
    # Create unique review_id
    pl.concat_str([pl.col('user_id'), pl.col('business_id'), pl.col('timestamp').cast(pl.Utf8)], separator='_').alias('review_id'),
    # Convert timestamp (milliseconds) to datetime
    pl.from_epoch(pl.col('timestamp'), time_unit='ms').alias('datetime')
])

print(f"Reviews prepared: {len(reviews_raw):,}")
print(f"Unique users: {reviews_raw['user_id'].n_unique():,}")
print(f"Unique products: {reviews_raw['business_id'].n_unique():,}")

### 2.2 Structure-Preserving Sampling

For fraud detection, we need to preserve co-occurrence structure. We sample **complete product histories** rather than random reviews to maintain reviewer overlap patterns essential for detecting coordinated campaigns.

In [ ]:
print("Applying structure-preserving sampling...")
print("This ensures reviewer co-occurrence patterns are maintained.\n")

# Analyze product review distribution
print("Analyzing product review counts...")
reviews_per_product = reviews_raw.group_by('business_id').agg(
    pl.count().alias('review_count'),
    pl.col('user_id').n_unique().alias('unique_reviewers')
)

print(f"Products with 1 review: {(reviews_per_product['review_count'] == 1).sum():,}")
print(f"Products with 2-9 reviews: {((reviews_per_product['review_count'] >= 2) & (reviews_per_product['review_count'] < 10)).sum():,}")
print(f"Products with 10-100 reviews: {((reviews_per_product['review_count'] >= 10) & (reviews_per_product['review_count'] <= 100)).sum():,}")
print(f"Products with 100+ reviews: {(reviews_per_product['review_count'] > 100).sum():,}")

# Filter to "Goldilocks zone" products (10-500 reviews, 8+ unique reviewers)
candidate_products = reviews_per_product.filter(
    (pl.col('review_count') >= 10) & 
    (pl.col('review_count') <= 500) &
    (pl.col('unique_reviewers') >= 8)
).sort('review_count', descending=True)

print(f"\nCandidate products (10-500 reviews, 8+ reviewers): {len(candidate_products):,}")

# Target ~200K reviews for good balance of speed and coverage
TARGET_REVIEWS = 200000
cumsum = candidate_products['review_count'].cum_sum()
n_products_needed = (cumsum <= TARGET_REVIEWS).sum()

selected_product_ids = set(
    candidate_products.head(min(n_products_needed + 500, len(candidate_products)))['business_id'].to_list()
)

print(f"Selected {len(selected_product_ids):,} products")

# Extract ALL reviews for selected products (preserves complete history)
reviews_df = reviews_raw.filter(
    pl.col('business_id').is_in(selected_product_ids)
)

print(f"\nFinal review count: {len(reviews_df):,}")
print(f"Unique users: {reviews_df['user_id'].n_unique():,}")
print(f"Unique products: {reviews_df['business_id'].n_unique():,}")

# Verify co-occurrence structure
print("\n--- STRUCTURE PRESERVATION CHECK ---")
user_review_counts = reviews_df.group_by('user_id').agg(pl.count().alias('n'))
users_with_multiple = (user_review_counts['n'] >= 2).sum()
print(f"Users with 2+ reviews: {users_with_multiple:,} ({users_with_multiple/len(user_review_counts)*100:.1f}%)")

# Clean up memory
del all_reviews, reviews_raw

In [ ]:
# Display sample reviews
print("Sample reviews:")
reviews_df.select(['review_id', 'user_id', 'business_id', 'stars', 'datetime', 'text', 'verified_purchase']).head(3)

### 2.3 Create User Statistics

In [ ]:
print("Creating user statistics...")

users_df = reviews_df.group_by('user_id').agg([
    pl.count().alias('review_count'),
    pl.col('stars').mean().alias('average_stars'),
    pl.col('verified_purchase').sum().alias('verified_purchases'),
    pl.col('helpful_vote').sum().alias('total_helpful_votes'),
])

print(f"Users in dataset: {len(users_df):,}")
print(f"\nUser statistics:")
print(users_df.describe())

## 3. Data Cleaning and Preprocessing

In [ ]:
print("Data quality check...\n")

print("Missing values in reviews:")
null_counts = reviews_df.null_count()
print(null_counts)

initial_count = len(reviews_df)

# Drop reviews with missing critical fields
reviews_df = reviews_df.drop_nulls(['text', 'stars'])

dropped = initial_count - len(reviews_df)
print(f"\nDropped {dropped} reviews with missing values ({dropped/initial_count*100:.2f}%)")
print(f"Remaining reviews: {len(reviews_df):,}")

In [ ]:
# Add temporal features
print("Processing dates...")

reviews_df = reviews_df.with_columns([
    pl.col('datetime').dt.year().alias('year'),
    pl.col('datetime').dt.month().alias('month'),
    pl.col('datetime').dt.day().alias('day'),
    pl.col('datetime').dt.weekday().alias('weekday'),
])

print("Date features added successfully!")
print(f"Date range: {reviews_df['datetime'].min()} to {reviews_df['datetime'].max()}")
reviews_df.select(['datetime', 'year', 'month', 'weekday']).head(3)

In [ ]:
# Add text length features
reviews_df = reviews_df.with_columns([
    pl.col('text').str.len_chars().alias('text_length'),
    pl.col('text').str.split(' ').list.len().alias('word_count')
])

print("Text features:")
print(reviews_df.select(['text_length', 'word_count']).describe())

### 3.1 Remove Extreme Outliers

In [ ]:
# Find bot-like users (>50 reviews in a single day)
print("Identifying extreme outliers...\n")

daily_reviews = reviews_df.group_by(['user_id', 'year', 'month', 'day']).agg(
    pl.count().alias('reviews_per_day')
)

extreme_users = daily_reviews.filter(pl.col('reviews_per_day') > 50)['user_id'].unique()

print(f"Users with >50 reviews/day: {len(extreme_users)}")

before_removal = len(reviews_df)
reviews_df = reviews_df.filter(~pl.col('user_id').is_in(extreme_users))
print(f"Reviews removed: {before_removal - len(reviews_df):,}")
print(f"Remaining reviews: {len(reviews_df):,}")

In [ ]:
# Cap review length at 99th percentile
length_99th = reviews_df['text_length'].quantile(0.99)
print(f"99th percentile text length: {length_99th:.0f} characters")

before_cap = len(reviews_df)
reviews_df = reviews_df.filter(pl.col('text_length') <= length_99th)
print(f"Reviews removed (too long): {before_cap - len(reviews_df):,}")
print(f"Remaining reviews: {len(reviews_df):,}")

## 4. Exploratory Data Analysis

### 4.1 Dataset Statistics

In [ ]:
print("=" * 60)
print("FINAL DATASET STATISTICS")
print("=" * 60)
print(f"Total reviews: {len(reviews_df):,}")
print(f"Unique users: {reviews_df['user_id'].n_unique():,}")
print(f"Unique products: {reviews_df['business_id'].n_unique():,}")
print(f"Date range: {reviews_df['datetime'].min()} to {reviews_df['datetime'].max()}")
print(f"\nVerified purchases: {reviews_df['verified_purchase'].sum():,} ({reviews_df['verified_purchase'].mean()*100:.1f}%)")
print(f"\nRating distribution:")
print(reviews_df['stars'].value_counts().sort('stars'))
print("=" * 60)

### 4.2 Reviews Per User Distribution

In [ ]:
reviews_per_user = reviews_df.group_by('user_id').agg(
    pl.count().alias('review_count')
).sort('review_count', descending=True)

print("Reviews per user statistics:")
print(reviews_per_user['review_count'].describe())

print("\nTop 10 most active users:")
print(reviews_per_user.head(10))

In [ ]:
# Figure 1: Distribution of reviews per user (log-log scale)
fig, ax = plt.subplots(figsize=(10, 6))

review_counts = reviews_per_user['review_count'].to_numpy()
bins = np.logspace(0, np.log10(review_counts.max()), 50)
ax.hist(review_counts, bins=bins, alpha=0.7, edgecolor='black')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Reviews per User', fontsize=12)
ax.set_ylabel('Number of Users', fontsize=12)
ax.set_title('Distribution of Reviews per User (log-log scale)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

ax.annotate('Heavy tail: potential fraudulent accounts', 
            xy=(review_counts.max() * 0.3, 10), fontsize=10, 
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))

plt.tight_layout()
plt.savefig(output_dir / 'fig1_reviews_per_user.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 1 saved to output/fig1_reviews_per_user.png")

### 4.3 Temporal Analysis - Review Bursts

In [ ]:
# Find products with suspicious burst patterns
reviews_weekly = reviews_df.with_columns(
    pl.col('datetime').dt.truncate('1w').alias('week')
).group_by(['business_id', 'week']).agg(
    pl.count().alias('weekly_reviews')
).sort(['business_id', 'week'])

burst_products = reviews_weekly.filter(pl.col('weekly_reviews') > 15)
print(f"Products with burst weeks (>15 reviews/week): {burst_products['business_id'].n_unique()}")

if len(burst_products) > 0:
    sample_product_id = burst_products['business_id'].first()
    print(f"Selected product for visualization: {sample_product_id}")
else:
    sample_product_id = reviews_df.group_by('business_id').agg(
        pl.count().alias('count')
    ).sort('count', descending=True)['business_id'].first()
    print(f"Using most-reviewed product: {sample_product_id}")

In [ ]:
# Figure 2: Temporal burst visualization
product_reviews = reviews_df.filter(
    pl.col('business_id') == sample_product_id
).sort('datetime')

daily_counts = product_reviews.group_by_dynamic('datetime', every='1d').agg(
    pl.count().alias('daily_reviews')
)

fig, ax = plt.subplots(figsize=(12, 6))

dates = daily_counts['datetime'].to_numpy()
counts = daily_counts['daily_reviews'].to_numpy()

ax.plot(dates, counts, linewidth=1.5, color='steelblue')
ax.fill_between(dates, counts, alpha=0.3, color='steelblue')

# Highlight burst periods
if len(counts) > 0:
    burst_threshold = counts.mean() + 2 * counts.std()
    burst_mask = counts > burst_threshold
    if burst_mask.any():
        ax.fill_between(dates, 0, counts.max(), where=burst_mask, 
                        alpha=0.3, color='orange', label='Suspicious burst')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Daily Review Count', fontsize=12)
ax.set_title(f'Review Activity Over Time\nProduct: {sample_product_id}', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'fig2_temporal_burst.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 2 saved to output/fig2_temporal_burst.png")

### 4.4 Rating Patterns

In [ ]:
# Analyze rating patterns per user
user_rating_stats = reviews_df.group_by('user_id').agg([
    pl.col('stars').mean().alias('avg_rating'),
    pl.col('stars').std().alias('rating_std'),
    pl.count().alias('num_reviews')
]).filter(pl.col('num_reviews') >= 5)

print(f"Users with 5+ reviews: {len(user_rating_stats):,}")
print("\nRating statistics:")
print(user_rating_stats.select(['avg_rating', 'rating_std']).describe())

In [ ]:
# Figure 3: Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall rating distribution
rating_counts = reviews_df['stars'].value_counts().sort('stars')
axes[0].bar(rating_counts['stars'].to_numpy(), rating_counts['count'].to_numpy(), 
            color='skyblue', edgecolor='black')
axes[0].set_xlabel('Star Rating', fontsize=12)
axes[0].set_ylabel('Number of Reviews', fontsize=12)
axes[0].set_title('Overall Rating Distribution', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Rating std per user (low std = suspicious)
std_values = user_rating_stats['rating_std'].drop_nulls().to_numpy()
axes[1].hist(std_values, bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(std_values.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {std_values.mean():.2f}')
axes[1].set_xlabel('Rating Standard Deviation', fontsize=12)
axes[1].set_ylabel('Number of Users', fontsize=12)
axes[1].set_title('User Rating Variability\n(Low std = always same rating)', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'fig3_rating_patterns.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 3 saved to output/fig3_rating_patterns.png")

### 4.5 Text Length Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

text_lengths = reviews_df['text_length'].to_numpy()
axes[0].hist(text_lengths, bins=50, color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[0].axvline(text_lengths.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {text_lengths.mean():.0f}')
axes[0].set_xlabel('Text Length (characters)', fontsize=12)
axes[0].set_ylabel('Number of Reviews', fontsize=12)
axes[0].set_title('Review Text Length Distribution', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

word_counts = reviews_df['word_count'].to_numpy()
axes[1].hist(word_counts, bins=50, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1].axvline(word_counts.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {word_counts.mean():.0f}')
axes[1].set_xlabel('Word Count', fontsize=12)
axes[1].set_ylabel('Number of Reviews', fontsize=12)
axes[1].set_title('Review Word Count Distribution', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'fig_text_length.png', dpi=300, bbox_inches='tight')
plt.show()
print("Text length figure saved")

### 4.6 Verified vs Unverified Purchase Analysis

In [ ]:
# Compare verified vs unverified purchases
print("Verified vs Unverified Purchase Analysis:")
print("="*50)

verified_stats = reviews_df.group_by('verified_purchase').agg([
    pl.count().alias('count'),
    pl.col('stars').mean().alias('avg_rating'),
    pl.col('text_length').mean().alias('avg_text_length'),
])
print(verified_stats)

# Unverified purchases often have inflated ratings (fraud signal)
verified_avg = reviews_df.filter(pl.col('verified_purchase') == True)['stars'].mean()
unverified_avg = reviews_df.filter(pl.col('verified_purchase') == False)['stars'].mean()
print(f"\nAverage rating (verified): {verified_avg:.2f}")
print(f"Average rating (unverified): {unverified_avg:.2f}")
print(f"Difference: {unverified_avg - verified_avg:.2f} stars")

## 5. Verify Co-occurrence Structure

In [ ]:
print("="*70)
print("CO-OCCURRENCE STRUCTURE VERIFICATION")
print("="*70)

# User activity distribution
rpu = reviews_df.group_by('user_id').agg(pl.count().alias('n'))
users_1 = (rpu['n'] == 1).sum()
users_2plus = (rpu['n'] >= 2).sum()
users_5plus = (rpu['n'] >= 5).sum()

print(f"\n[User Activity Distribution]")
print(f"  Users with 1 review: {users_1:,} ({users_1/len(rpu)*100:.1f}%)")
print(f"  Users with 2+ reviews: {users_2plus:,} ({users_2plus/len(rpu)*100:.1f}%)")
print(f"  Users with 5+ reviews: {users_5plus:,} ({users_5plus/len(rpu)*100:.1f}%)")

# Product-level transactions for FP-Growth
product_transactions = reviews_df.group_by('business_id').agg(
    pl.col('user_id').n_unique().alias('n_reviewers')
)
products_2plus = (product_transactions['n_reviewers'] >= 2).sum()
products_5plus = (product_transactions['n_reviewers'] >= 5).sum()

print(f"\n[Product-Level Transactions for FP-Growth]")
print(f"  Products with 2+ reviewers: {products_2plus:,}")
print(f"  Products with 5+ reviewers: {products_5plus:,}")

# Verdict
print("\n" + "="*70)
if users_2plus >= 1000 and products_2plus >= 500:
    print("STRUCTURE PRESERVED: Data is suitable for fraud detection!")
else:
    print("WARNING: May need to adjust sampling parameters")
print("="*70)

## 6. Save Processed Data

In [ ]:
print("Saving processed data...")

# Save reviews
reviews_df.write_parquet('processed_reviews.parquet')
print(f"Saved {len(reviews_df):,} reviews to processed_reviews.parquet")

# Create and save products summary
products_df = reviews_df.group_by('business_id').agg([
    pl.count().alias('review_count'),
    pl.col('stars').mean().alias('avg_rating'),
    pl.col('verified_purchase').mean().alias('verified_rate'),
]).sort('review_count', descending=True)

products_df.write_parquet('processed_businesses.parquet')
print(f"Saved {len(products_df):,} products to processed_businesses.parquet")

# Save users
users_df.write_parquet('processed_users.parquet')
print(f"Saved {len(users_df):,} users to processed_users.parquet")

## 7. Summary Statistics for Report

In [ ]:
print("="*70)
print("SUMMARY STATISTICS FOR LATEX REPORT")
print("="*70)

print(f"\n[Dataset Statistics]")
print(f"Total reviews: {len(reviews_df):,}")
print(f"Unique users: {reviews_df['user_id'].n_unique():,}")
print(f"Unique products: {reviews_df['business_id'].n_unique():,}")
print(f"Date range: {reviews_df['year'].min()} to {reviews_df['year'].max()}")

print(f"\n[Data Quality]")
print(f"Missing values dropped: {dropped/initial_count*100:.2f}%")

print(f"\n[User Activity]")
print(f"Mean reviews per user: {rpu['n'].mean():.1f}")
print(f"Median reviews per user: {rpu['n'].median():.1f}")
print(f"Max reviews per user: {rpu['n'].max()}")

print(f"\n[Text Statistics]")
print(f"Average text length: {reviews_df['text_length'].mean():.0f} characters")
print(f"Average word count: {reviews_df['word_count'].mean():.0f} words")

print(f"\n[Rating Distribution]")
for star in sorted(reviews_df['stars'].unique().to_list()):
    count = (reviews_df['stars'] == star).sum()
    pct = count / len(reviews_df) * 100
    print(f"  {star}-star: {count:,} ({pct:.1f}%)")

print(f"\n[Amazon-Specific]")
verified_pct = reviews_df['verified_purchase'].mean() * 100
print(f"Verified purchases: {verified_pct:.1f}%")
print(f"Unverified purchases: {100-verified_pct:.1f}%")

print("\n" + "="*70)

## EDA Complete!

Key findings:
1. **Heavy-tailed distribution**: Most users write few reviews, but some write many (potential fraud)
2. **Temporal bursts**: Some products show suspicious spikes in review activity
3. **Rating extremes**: Some users always give 5-star or 1-star ratings
4. **Verified vs Unverified**: Unverified purchases may have different rating patterns

**Next step**: Run `analysis.ipynb` to implement the 5-stage fraud detection pipeline!